# 01 - First Look: Sampling the Raw Data

**Stage 1 of the project.**

This notebook is the first thing anyone opening this repository should run.
Its only job is to answer one question: *"what condition is the raw data
actually in?"* -- before we touch a single value. We start with a quick,
visual peek at rows that are obviously off, then follow up with a
systematic pass over every table.

### Why this step exists
In a real factory, data does not arrive clean. It comes out of MES
(Manufacturing Execution System) exports, shift-handover spreadsheets, and
paper travelers typed up after the fact. Different operators type the same
thing in different ways, fields get left blank, and copy-paste mistakes
duplicate rows. Before any KPI (OEE, Cp/Cpk, MTBF...) can be trusted, we
have to know exactly how dirty the input is -- otherwise a wrong number in
the data becomes a wrong number in the dashboard, silently.

We do **not** fix anything in this notebook -- that is Stage 3
(`02_data_cleaning_production.ipynb`). Keeping "look" and "fix" in
separate notebooks is a deliberate habit: it forces you to write down
every problem you found *before* you start solving them, so nothing gets
fixed by accident without being noticed and explained.


In [1]:
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

RAW = '../../datasets/raw'
DIM = '../../datasets/dim'


## 0. A first peek: what does "bad data" actually look like here?

Before running anything systematic, it's worth just *looking* at a
handful of rows that are visibly off -- this is the fastest way to
build intuition for what Stage 3 will need to handle.


In [2]:
prod_peek = pd.read_csv(f'{RAW}/fact_production_raw.csv', encoding='utf-8-sig')
downtime_peek = pd.read_csv(f'{RAW}/fact_downtime_raw.csv', encoding='utf-8-sig')

print("-- Inconsistent category casing (same process, typed differently) --")
display(prod_peek[prod_peek['Process'].isin(['injection molding', 'INJECTION MOLDING', '  Injection Molding  '])].head(6))

print("-- A quantity that should never be negative --")
display(prod_peek[prod_peek['RejectedQty'] < 0].head(6))

print("-- Fully duplicated rows --")
display(prod_peek[prod_peek.duplicated(keep=False)].sort_values('WorkOrder').head(6))

print("-- Placeholder nulls hiding in a text field ('-', '/', a lone space) --")
display(downtime_peek[downtime_peek['MaintenanceTechnician'].isin(['-', '--', '/', '//', ' ', '  '])].head(6))


-- Inconsistent category casing (same process, typed differently) --


,Date,Process,MachineId,ToolId,WorkOrder,ProductId,ProductBatch,StartTime,EndTime,PlannedQty,ProducedQty,RejectedQty,OperatorId
5,2026-01-02,injection molding,IM-005,M-INJ-008,WO-4269,TF-008-PP-PCR-28410,26002002700,11:08:00,18:33:00,5684,5684,185,OP-INJ-003
12,2026-04-15,Injection Molding,IM-002,RS-HF-002,WO-2895,TR-002-HDPE-24410,26105008960,00:19:00,11:12:00,5306,5324,168,OP-INJ-002
24,2026-05-21,Injection Molding,IM-001,M-INJ-001,WO-2440,TR-001-PP-24410,26141004410,07:59:00,23:12:00,18053,18053,240,OP-INJ-003
32,2026-02-16,Injection Molding,IM-006,M-INJ-006,WO-4846,TF-006-HDPE-24415,26047008470,22:13:00,11:25:00,5259,5402,312,OP-INJ-001
36,2026-04-27,injection molding,IM-005,M-INJ-005,WO-4432,TF-005-PP-24410,26117004330,21:46:00,03:38:00,4638,4738,88,OP-INJ-001
41,2025-10-21,injection molding,IM-002,M-INJ-002,WO-2646,TR-002-PETG-24410,25294006470,00:56:00,14:06:00,9653,9934,488,OP-INJ-002


-- A quantity that should never be negative --


,Date,Process,MachineId,ToolId,WorkOrder,ProductId,ProductBatch,StartTime,EndTime,PlannedQty,ProducedQty,RejectedQty,OperatorId
15,2025-11-28,blow molding,ISBM-006,M-SOP-014,WO-7792,FR-014-PP-PCR-500,25332007930,19:04:00,03:23:00,4738,4886,-107,OP-SOP-002
388,2026-03-31,Hot Foil Stamping,HF-001,RS-HF-002,WO-1356,FR-015-PETG-500,26090003570,10:27:00,05:50:00,12074,12198,-16,OP-HF-001
499,2026-01-06,INJECTION MOLDING,IM-002,M-INJ-002,WO-2758,TR-002-HDPE-24410,26006007590,08:47:00,15:16:00,4900,4942,-153,OP-INJ-001
682,2026-02-10,Injection Molding,IM-006,M-INJ-009,WO-4836,TF-009-HDPE-28415,26041008370,15:08:00,07:22:00,9381,9487,-580,OP-INJ-002
1030,2025-11-13,blow molding,ISBM-008,M-SOP-019,WO-8771,FR-019-rPET-600,25317007720,11:14:00,18:49:00,4206,4307,-76,OP-SOP-001
1145,2025-08-06,Injection Molding,IM-006,M-INJ-009,WO-4573,TF-009-HDPE-28415,25218005740,19:44:00,07:00:00,6625,6782,-391,OP-INJ-003


-- Fully duplicated rows --


,Date,Process,MachineId,ToolId,WorkOrder,ProductId,ProductBatch,StartTime,EndTime,PlannedQty,ProducedQty,RejectedQty,OperatorId
5032,2026-01-15,INJECTION MOLDING,IM-001,M-INJ-001,WO-2270,TR-001-HDPE-24410,26015002710,19:39:00,07:21:00,13993,14143,146,AUX-INJ-001
2768,2026-01-15,INJECTION MOLDING,IM-001,M-INJ-001,WO-2270,TR-001-HDPE-24410,26015002710,19:39:00,07:21:00,13993,14143,146,AUX-INJ-001
4236,2026-03-06,Injection Molding,IM-001,M-INJ-001,WO-2339,TR-001-PP-24410,26065003400,17:47:00,01:06:00,8643,8643,237,OP-INJ-002
2904,2026-03-06,Injection Molding,IM-001,M-INJ-001,WO-2339,TR-001-PP-24410,26065003400,17:47:00,01:06:00,8643,8643,237,OP-INJ-002
5667,2026-04-01,Injection Molding,IM-001,M-INJ-001,WO-2375,TR-001-PP-24410,26091003760,07:28:00,19:17:00,13084,13084,326,AUX-INJ-001
6446,2026-04-01,Injection Molding,IM-001,M-INJ-001,WO-2375,TR-001-PP-24410,26091003760,07:28:00,19:17:00,13084,13084,326,AUX-INJ-001


-- Placeholder nulls hiding in a text field ('-', '/', a lone space) --


,Date,Process,MachineId,Shift,StoppageStartTime,StoppageEndTime,PlannedStoppage,StoppageReason,MaintenanceTeam,MaintenanceTechnician
41,2025-09-19,Blow Molding,ISBM-002,Shift 2,20:28:00,21:34:00,Yes,Planned Preventive Maintenance,Mechanical,--
219,2026-04-23,Blow Molding,ISBM-005,Shift 2,20:42:00,22:11:00,No,Mechanical Failure (Breakage/Jam),Mechanical,
655,2025-08-20,Injection Molding,IM-004,Shift 3,02:00:00,03:20:00,Yes,"Meal Break (Shift 3 - no relief crew, 10min sh...",N/A (Operations),--
698,2026-01-01,blow molding,ISBM-004,shift 1,11:17:00,11:46:00,No,Quality Adjustment (Rejects),Mechanical,
1001,2025-11-04,Blow Molding,ISBM-006,Shift 1,12:19:00,13:06:00,Yes,Mold Change / Setup,Mechanical,--
1067,2025-08-13,Blow Molding,ISBM-006,SHIFT 3,04:51:00,05:29:00,No,Utilities Shortage (Compressed Air),Utilities,/


That's the flavor of it. Now the systematic pass: for every raw table in
`datasets/raw/` and `datasets/dim/` we run the classic first-look trio:

| Function | What it tells us |
|---|---|
| `df.head(30)` | What the data actually looks like, row by row |
| `df.info()` | Column names, data types, and non-null counts in one shot |
| `df.isna().sum()` | Exactly how many missing values are in each column |


## 1. Fact tables (production, downtime, material consumption, plan)

These four tables are the operational backbone of the project: every
work order, every machine stoppage, and every raw-material lot consumed
on the shop floor.


In [3]:
fact_files = {
    'fact_production_plan_raw': f'{RAW}/fact_production_plan_raw.csv',
    'fact_production_raw': f'{RAW}/fact_production_raw.csv',
    'fact_downtime_raw': f'{RAW}/fact_downtime_raw.csv',
    'fact_material_consumption_raw': f'{RAW}/fact_material_consumption_raw.csv',
}

raw_tables = {name: pd.read_csv(path, encoding='utf-8-sig') for name, path in fact_files.items()}
for name, df in raw_tables.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")


fact_production_plan_raw: 9,128 rows x 10 columns
fact_production_raw: 9,137 rows x 13 columns
fact_downtime_raw: 21,874 rows x 10 columns
fact_material_consumption_raw: 23,504 rows x 10 columns


In [4]:
# head(30) for every fact table -- what does the data actually look like?
for name, df in raw_tables.items():
    print(f"\n{'='*100}\n{name} -- first 30 rows\n{'='*100}")
    display(df.head(30))



fact_production_plan_raw -- first 30 rows


,Date,Process,MachineId,ToolId,WorkOrder,PlannedQty,StartTime,EndTime,PlannedHours,ProductId
0,2025-09-09,Blow Molding,ISBM-002,M-SOP-008,WO-5651,8830,18:23:00,09:25:00,15.03,FR-008-HDPE-PCR-350
1,2025-08-12,Injection Molding,IM-005,M-INJ-008,WO-4074,16335,22:46:00,20:27:00,21.68,TF-008-HDPE-28410
2,2025-10-08,BLOW MOLDING,ISBM-003,M-SOP-025,WO-6199,4841,13:14:00,06:26:00,17.19,FR-025-PET-1000
3,2026-01-05,Blow Molding,ISBM-005,M-SOP-010,WO-7328,10067,21:51:00,15:15:00,17.41,FR-010-PET-400
4,2026-04-09,Injection Molding,IM-003,M-INJ-003,WO-3382,19922,10:31:00,00:05:00,13.57,TR-003-PP-24415
5,2025-11-04,Injection Molding,IM-001,M-INJ-001,WO-2170,7549,17:08:00,00:02:00,6.89,TR-001-PP-PCR-24410
6,2025-12-26,blow molding,ISBM-005,M-SOP-023,WO-7316,4061,13:40:00,03:37:00,13.94,FR-023-rPET-1000
7,2026-03-27,Blow Molding,ISBM-007,M-SOP-015,WO-8456,8926,02:33:00,23:34:00,21.02,FR-015-HDPE-PCR-500
8,2025-09-20,Blow Molding,ISBM-006,M-SOP-014,WO-7693,2899,00:45:00,06:00:00,5.25,FR-014-rPET-500
9,2025-12-17,blow molding,ISBM-006,M-SOP-011,WO-7815,8745,12:20:00,03:32:00,15.21,FR-011-PET-400



fact_production_raw -- first 30 rows


,Date,Process,MachineId,ToolId,WorkOrder,ProductId,ProductBatch,StartTime,EndTime,PlannedQty,ProducedQty,RejectedQty,OperatorId
0,2025-12-04,Hot Foil Stamping,HF-002,RS-HF-001,WO-1703,FR-008-LDPE-350,25338007040,13:39:00,19:40:00,3417,3511,6,OP-HF-002
1,2026-05-19,Blow Molding,ISBM-007,M-SOP-020,WO-8533,FR-020-rPET-750,26139005340,21:28:00,12:53:00,6341,6442,168,OP-SOP-001
2,2025-08-12,Blow Molding,ISBM-003,M-SOP-006,WO-6123,FR-006-rPET-330,25224001240,01:09:00,17:06:00,6970,7202,130,OP-SOP-001
3,2025-07-30,Blow Molding,ISBM-002,M-SOP-008,WO-5594,FR-008-PETG-350,25211005950,01:22:00,17:26:00,9042,9042,281,OP-SOP-001
4,2025-12-01,Injection Molding,IM-002,M-INJ-002,WO-2703,TR-002-PETG-24410,25335007040,06:00:00,23:07:00,13179,13421,378,AUX-INJ-001
5,2026-01-02,injection molding,IM-005,M-INJ-008,WO-4269,TF-008-PP-PCR-28410,26002002700,11:08:00,18:33:00,5684,5684,185,OP-INJ-003
6,2025-09-18,Blow Molding,ISBM-006,M-SOP-018,WO-7691,FR-018-PP-600,25261006920,09:42:00,07:38:00,12390,12390,255,---
7,2025-09-08,Blow Molding,ISBM-003,M-SOP-009,WO-6158,FR-009-HDPE-PCR-350,25251001590,06:29:00,22:31:00,7101,7199,120,NaN
8,2026-01-28,Blow Molding,ISBM-004,M-SOP-004,WO-6861,FR-004-rPET-330,26028008620,06:04:00,19:38:00,7740,7867,112,OP-SOP-004
9,2026-04-10,Blow Molding,ISBM-004,M-SOP-029,WO-6954,FR-029-MDPE-5000,26100009550,21:04:00,16:16:00,1735,1760,194,OP-SOP-001



fact_downtime_raw -- first 30 rows


,Date,Process,MachineId,Shift,StoppageStartTime,StoppageEndTime,PlannedStoppage,StoppageReason,MaintenanceTeam,MaintenanceTechnician
0,2026-03-02,INJECTION MOLDING,IM-003,Shift 3,22:11:00,22:27:00,No,Raw Material Shortage,N/A (Operations),N/A (Operador de linha)
1,2025-07-25,Screen Printing,SS-001,shift 1,10:56:00,11:31:00,No,Mechanical Failure (Roller/Squeegee),Mechanical,Sandra Reis
2,2026-01-12,Injection Molding,IM-003,Shift 1,07:10:00,07:31:00,No,Operator Unavailable,N/A (Operations),N/A (Operador de linha)
3,2025-07-21,Blow Molding,ISBM-007,Shift 2,20:44:00,21:01:00,No,Quality Adjustment (Rejects),Mechanical,José Pinto
4,2025-11-21,injection molding,IM-005,Shift 3,04:23:00,04:55:00,No,Quality Adjustment (Rejects),Mechanical,José Pinto
5,2025-07-28,Screen Printing,SS-002,SHIFT 1,11:41:00,11:52:00,No,Color Registration Adjustment,Mechanical,Sandra Reis
6,2025-08-04,Injection Molding,IM-004,Shift 3,02:00:00,03:20:00,Yes,"Meal Break (Shift 3 - no relief crew, 10min sh...",N/A (Operations),N/A (Operador de linha)
7,2026-05-26,Blow Molding,ISBM-007,shift 3,22:44:00,23:08:00,No,Utilities Shortage (Compressed Air),Utilities,Sandra Reis
8,2025-09-01,Hot Foil Stamping,HF-001,Shift 1,09:35:00,11:04:00,Yes,Planned Preventive Maintenance,Mechanical,Sandra Reis
9,2025-10-31,BLOW MOLDING,ISBM-007,Shift 1,10:25:00,10:44:00,No,Utilities Shortage (Compressed Air),Utilities,Sandra Reis



fact_material_consumption_raw -- first 30 rows


,RecordSeq,Date,Process,MachineId,Shift,WorkOrder,MaterialId,MaterialLot,StartWeightKg,EndWeightKg
0,14577,2026-03-02,Hot Foil Stamping,HF-001,Shift 1,WO-1314,COR-001,COL-COR-001-107,59.11,34.24
1,4737,2025-09-30,INJECTION MOLDING,IM-003,SHIFT 3,WO-3119,COR-007,COL-COR-007-605,390.89,223.88
2,12421,2026-01-27,Screen Printing,SS-001,Shift 3,WO-9360,COR-003,COL-COR-003-2297,373.20,275.14
3,7066,2025-11-05,Blow Molding,ISBM-003,Shift 1,WO-6239,COR-008,COL-COR-008-3402,12.73,10.47
4,21517,2026-06-17,Blow Molding,ISBM-005,Shift 3,WO-7554,COR-005,COL-COR-005-1833,40.99,20.79
5,3614,2025-09-12,Injection Molding,IM-005,Shift 3,WO-4116,COR-003,COL-COR-003-680,231.61,162.45
6,23038,2026-07-10,BLOW MOLDING,ISBM-001,Shift 2,WO-5568,COR-003,COL-COR-003-1412,19.94,10.42
7,10793,2026-01-01,Blow Molding,ISBM-008,Shift 3,WO-8833,COR-006,COL-COR-006-1286,46.97,27.57
8,4892,2025-10-02,Hot Foil Stamping,HF-002,SHIFT 3,WO-1611,COR-001,COL-COR-001-190,23.61,18.64
9,17091,2026-04-09,Blow Molding,ISBM-002,Shift 3,WO-5947,COR-008,COL-COR-008-2946,37.62,17.55


In [5]:
# info() -- column names, dtypes, non-null counts
for name, df in raw_tables.items():
    print(f"\n{'='*100}\n{name}.info()\n{'='*100}")
    df.info()



fact_production_plan_raw.info()
<class 'pandas.DataFrame'>
RangeIndex: 9128 entries, 0 to 9127
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          9128 non-null   str    
 1   Process       9128 non-null   str    
 2   MachineId     9128 non-null   str    
 3   ToolId        9128 non-null   str    
 4   WorkOrder     9128 non-null   str    
 5   PlannedQty    9128 non-null   int64  
 6   StartTime     9128 non-null   str    
 7   EndTime       9128 non-null   str    
 8   PlannedHours  9128 non-null   float64
 9   ProductId     9128 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 713.3 KB

fact_production_raw.info()
<class 'pandas.DataFrame'>
RangeIndex: 9137 entries, 0 to 9136
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Date          9137 non-null   str  
 1   Process       9137 non-null   str  
 2   Ma

In [6]:
# isna().sum() -- exact missing-value count per column
for name, df in raw_tables.items():
    print(f"\n{'='*100}\n{name} -- missing values per column\n{'='*100}")
    print(df.isna().sum().to_string())



fact_production_plan_raw -- missing values per column
Date            0
Process         0
MachineId       0
ToolId          0
WorkOrder       0
PlannedQty      0
StartTime       0
EndTime         0
PlannedHours    0
ProductId       0

fact_production_raw -- missing values per column
Date             0
Process          0
MachineId        0
ToolId           0
WorkOrder        0
ProductId        0
ProductBatch     0
StartTime        0
EndTime          0
PlannedQty       0
ProducedQty      0
RejectedQty      0
OperatorId      26

fact_downtime_raw -- missing values per column
Date                      0
Process                   0
MachineId                 0
Shift                     0
StoppageStartTime         0
StoppageEndTime           0
PlannedStoppage           0
StoppageReason            0
MaintenanceTeam           0
MaintenanceTechnician    64

fact_material_consumption_raw -- missing values per column
RecordSeq         0
Date              0
Process           0
MachineId         0


## 2. Quality-control fact tables

Eight tables coming out of the quality lab / LIMS system: variable (X-bar/R)
measurements, attribute (AQL) inspections, and lot disposition decisions,
for the three decorated/molded product families -- caps, bottles, ink
(screen printing).


In [7]:
cq_files = {
    'fact_cap_inspection_variable_cq_raw': f'{RAW}/fact_cap_inspection_variable_cq_raw.csv',
    'fact_cap_attribute_inspection_cq_raw': f'{RAW}/fact_cap_attribute_inspection_cq_raw.csv',
    'fact_cap_disposition_lot_cq_raw': f'{RAW}/fact_cap_disposition_lot_cq_raw.csv',
    'fact_bottle_inspection_variables_cq_raw': f'{RAW}/fact_bottle_inspection_variables_cq_raw.csv',
    'fact_bottle_attribute_inspection_cq_raw': f'{RAW}/fact_bottle_attribute_inspection_cq_raw.csv',
    'fact_bottle_disposition_lot_cq_raw': f'{RAW}/fact_bottle_disposition_lot_cq_raw.csv',
    'fact_ink_attribute_inspection_cq_raw': f'{RAW}/fact_ink_attribute_inspection_cq_raw.csv',
    'fact_ink_disposition_lot_cq_raw': f'{RAW}/fact_ink_disposition_lot_cq_raw.csv',
}
cq_tables = {name: pd.read_csv(path, encoding='utf-8-sig') for name, path in cq_files.items()}
for name, df in cq_tables.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")


fact_cap_inspection_variable_cq_raw: 83,947 rows x 33 columns
fact_cap_attribute_inspection_cq_raw: 9,924 rows x 23 columns
fact_cap_disposition_lot_cq_raw: 1,654 rows x 22 columns
fact_bottle_inspection_variables_cq_raw: 124,580 rows x 27 columns
fact_bottle_attribute_inspection_cq_raw: 17,520 rows x 21 columns
fact_bottle_disposition_lot_cq_raw: 2,190 rows x 20 columns
fact_ink_attribute_inspection_cq_raw: 4,456 rows x 26 columns
fact_ink_disposition_lot_cq_raw: 557 rows x 22 columns


In [8]:
for name, df in cq_tables.items():
    print(f"\n{'='*100}\n{name} -- first 30 rows\n{'='*100}")
    display(df.head(30))



fact_cap_inspection_variable_cq_raw -- first 30 rows


,ProductBatch,WorkOrder,ProductionDate,Shift,MachineId,MoldId,CapId,Material,CapType,Characteristic,Equipment,Standard,Unit,InspectionDateTime,SampleGroup,LSL,Nominal,USL,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,XBar,RangeR,StdDevS,GroupResult,Inspector
0,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 09:34:00.000,1,10.3,12.05,13.8,12.155,11.954,11.738,11.891,11.703,12.071,12.519,11.878,11.833,12.221,11.996,0.816,0.250,Conforming,Elena Santos
1,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,SHIFT 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 10:04:00.000,2,10.3,12.05,13.8,12.087,11.724,12.040,12.293,11.580,11.890,11.385,11.599,11.405,11.968,11.797,0.909,0.306,Conforming,Elena Santos
2,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 10:34:00.000,3,10.3,12.05,13.8,12.145,12.105,11.985,11.169,11.861,12.033,12.090,11.514,11.883,11.708,11.849,0.976,0.309,Conforming,Elena Santos
3,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,SHIFT 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 11:04:00.000,4,10.3,12.05,13.8,12.421,11.767,12.039,12.360,11.846,12.011,12.089,12.072,11.621,12.077,12.030,0.800,0.245,Conforming,Elena Santos
4,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 11:34:00.000,5,10.3,12.05,13.8,11.508,12.351,12.092,11.825,12.750,12.317,11.630,12.076,12.252,11.984,12.079,1.242,0.367,Conforming,Elena Santos
5,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 12:04:00.000,6,10.3,12.05,13.8,12.027,12.284,12.553,11.814,12.121,11.888,12.095,11.634,11.847,11.981,12.024,0.919,0.260,Conforming,Elena Santos
6,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 12:34:00.000,7,10.3,12.05,13.8,12.451,11.587,11.772,12.276,11.353,11.888,12.016,12.490,12.291,11.935,12.006,1.137,0.375,Conforming,Elena Santos
7,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 13:04:00.000,8,10.3,12.05,13.8,11.962,12.583,11.900,11.944,12.173,12.008,11.981,11.660,12.046,11.895,12.015,0.923,0.238,Conforming,Elena Santos
8,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 1,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 13:34:00.000,9,10.3,12.05,13.8,12.279,12.042,12.284,11.931,12.418,12.048,12.254,11.598,12.171,11.459,12.048,0.959,0.310,Conforming,Elena Santos
9,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,SHIFT 2,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Weight,Scale 0.001 g,ISO 3951,g,2026-01-01 14:04:00.000,10,10.3,12.05,13.8,11.943,11.735,12.107,12.836,11.759,11.832,12.122,12.223,11.988,11.978,12.052,1.101,0.318,Conforming,Elena Santos



fact_cap_attribute_inspection_cq_raw -- first 30 rows


,ProductBatch,WorkOrder,ProductionDate,Shift,MachineId,MoldId,CapId,Material,CapType,Characteristic,Class,AQL,Standard,InspectionLevel,LotSize,CodeLetter,SampleSize,AcceptanceNumber,RejectionNumber,DefectsFound,LotDecision,InspectionDateTime,Inspector
0,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 2,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Thread,Critical,0.10,ISO 2859-1,II,23450,M,315,0,1,0,Approved,2026-01-01 19:34:14.862,Elena Santos
1,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,shift 3,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Sealing,Critical,0.10,ISO 2859-1,II,23450,M,315,0,1,0,Approved,2026-01-02 01:30:59.992,Elena Santos
2,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 3,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Flash,Major,0.65,ISO 2859-1,II,23450,M,315,3,4,7,Rejected,2026-01-02 03:10:38.033,Elena Santos
3,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,Shift 2,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Short Shot,Major,0.65,ISO 2859-1,II,23450,M,315,3,4,0,Approved,2026-01-01 19:32:42.389,Elena Santos
4,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,shift 3,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Stain,Minor,1.50,ISO 2859-1,II,23450,M,315,14,15,0,Approved,2026-01-02 00:51:23.247,Elena Santos
5,LOTE-M-INJ-001-2490,WO-2248,2026-01-01,SHIFT 3,IM-001,M-INJ-001,TR-001-PETG-24410,PETG,Screw Cap,Color,Minor,1.50,ISO 2859-1,II,23450,M,315,14,15,1,Approved,2026-01-01 22:14:36.942,Elena Santos
6,LOTE-M-INJ-002-7520,WO-2751,2026-01-01,SHIFT 3,IM-002,M-INJ-002,TR-002-PP-PCR-24410,PP-PCR,Screw Cap,Thread,Critical,0.10,ISO 2859-1,II,6610,L,200,0,1,0,Approved,2026-01-01 22:45:11.449,Ana Silva
7,LOTE-M-INJ-002-7520,WO-2751,2026-01-01,Shift 3,IM-002,M-INJ-002,TR-002-PP-PCR-24410,PP-PCR,Screw Cap,Sealing,Critical,0.10,ISO 2859-1,II,6610,L,200,0,1,0,Approved,2026-01-01 22:35:06.825,Ana Silva
8,LOTE-M-INJ-002-7520,WO-2751,2026-01-01,Shift 2,IM-002,M-INJ-002,TR-002-PP-PCR-24410,PP-PCR,Screw Cap,Flash,Major,0.65,ISO 2859-1,II,6610,L,200,2,3,0,Approved,2026-01-01 21:57:27.728,Ana Silva
9,LOTE-M-INJ-002-7520,WO-2751,2026-01-01,Shift 3,IM-002,M-INJ-002,TR-002-PP-PCR-24410,PP-PCR,Screw Cap,Short Shot,Major,0.65,ISO 2859-1,II,6610,L,200,2,3,0,Approved,2026-01-01 22:16:43.698,Ana Silva



fact_cap_disposition_lot_cq_raw -- first 30 rows


...

,ProductBatch,WorkOrder,ProductionDate,Shift,MachineId,MoldId,BottleId,LotSize,CodeLetter,SampleSize,CriticalDefects,MajorDefects,MinorDefects,TotalSampleDefects,VariablesDecision,AttributesDecision,FinalLotDecision,LotDecisionDateTime,Remarks,Inspector
0,LOTE-M-SOP-007-2950,WO-5294,2026-01-01,SHIFT 3,ISBM-001,M-SOP-007,FR-007-PP-350,3554,L,200,0,0,0,0,Conforming,Approved,Approved,2026-01-01 08:47:30.771,—,Beatriz Costa
1,LOTE-M-SOP-001-2960,WO-5295,2026-01-01,Shift 1,ISBM-001,M-SOP-001,FR-001-HDPE-300,10738,M,315,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-02 04:30:04.667,—,Diogo Ferreira
2,LOTE-M-SOP-008-8110,WO-5810,2026-01-01,shift 3,ISBM-002,M-SOP-008,FR-008-PET-350,9293,L,200,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-01 21:19:28.900,—,Elena Santos
3,LOTE-M-SOP-008-8120,WO-5811,2026-01-01,Shift 2,ISBM-002,M-SOP-008,FR-008-PET-350,11290,M,315,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-02 17:48:03.332,—,Elena Santos
4,LOTE-M-SOP-009-3240,WO-6323,2026-01-01,Shift 3,ISBM-003,M-SOP-009,FR-009-HDPE-PCR-350,6622,L,200,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-01 19:11:56.228,—,Ana Silva
5,LOTE-M-SOP-009-3250,WO-6324,2026-01-01,Shift 2,ISBM-003,M-SOP-009,FR-009-HDPE-PCR-350,4580,L,200,0,3,0,3,Conforming,Approved,Approved,2026-01-02 05:37:08.248,Flash:3(Re=3),Ana Silva
6,LOTE-M-SOP-029-8230,WO-6822,2026-01-01,Shift 2,ISBM-004,M-SOP-029,FR-029-MDPE-5000,476,H,50,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-01 22:54:24.701,—,Beatriz Costa
7,LOTE-M-SOP-026-8240,WO-6823,2026-01-01,Shift 3,ISBM-004,M-SOP-026,FR-026-HDPE-4000,1625,K,125,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-02 08:14:39.561,—,Ana Silva
8,LOTE-M-SOP-013-3230,WO-7322,2026-01-01,shift 1,ISBM-005,M-SOP-013,FR-013-LDPE-500,6578,L,200,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-02 01:11:08.707,—,Beatriz Costa
9,LOTE-M-SOP-018-8340,WO-7833,2026-01-01,shift 1,ISBM-006,M-SOP-018,FR-018-PP-PCR-600,12594,M,315,0,0,0,0,Nonconformance Detected,Approved,Approved,2026-01-02 05:30:55.146,—,Elena Santos



fact_ink_attribute_inspection_cq_raw -- first 30 rows


,PrintLot,ProductBatch,WorkOrder,ProductionDate,Shift,MachineId,PrintToolId,BottleId,ColorCount,ColorsUsed,Characteristic,Class,AQL,Standard,InspectionLevel,Frequency,Equipment,LotSize,CodeLetter,SampleSize,AcceptanceNumber,RejectionNumber,DefectsFound,LotDecision,InspectionDateTime,Inspector
0,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 2,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Registration,Critical,0.65,ISO 2859-1,II,Per lot,Registration Gauge,19128,M,315,3,4,0,Approved,2026-01-01 15:02:25.584,Elena Santos
1,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,shift 1,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Centering,Critical,0.65,ISO 2859-1,II,Every 30 min,Registration Gauge,19128,M,315,3,4,1,Approved,2026-01-01 12:48:22.985,Elena Santos
2,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 1,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Coverage,Major,0.65,ISO 2859-1,II,Every 30 min,D65 Light Booth,19128,M,315,3,4,0,Approved,2026-01-01 10:09:43.076,Elena Santos
3,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 2,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Color,Minor,1.50,ISO 2859-1,II,Every 30 min,Colorimeter,19128,M,315,14,15,0,Approved,2026-01-01 21:21:34.005,Elena Santos
4,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 1,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Smudge,Minor,1.50,ISO 2859-1,II,Continuous,D65 Light Booth,19128,M,315,14,15,0,Approved,2026-01-01 06:20:23.561,Elena Santos
5,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,shift 1,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Pinholes,Minor,1.50,ISO 2859-1,II,Continuous,Lupa 10x,19128,M,315,14,15,0,Approved,2026-01-01 08:33:00.829,Elena Santos
6,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 1,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Adhesion,Critical,0.10,ISO 2859-1,II,Per lot (fixed),ASTM D3359 Kit,19128,—,3,0,1,0,Approved,2026-01-01 07:10:31.124,Elena Santos
7,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 3,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",Cure,Critical,0.10,ISO 2859-1,II,Per lot (fixed),MEK Rub Test,19128,—,3,0,1,0,Approved,2026-01-01 04:05:06.730,Elena Santos
8,LOTE-SK-M-SOP-013-8350,LOTE-M-SOP-013-8350,WO-9834,2026-01-01,Shift 1,SS-002,SK-FR-013-MDPE-500-2C,FR-013-MDPE-500,2,"COR-002, COR-005",Registration,Critical,0.65,ISO 2859-1,II,Per lot,Registration Gauge,7562,L,200,2,3,0,Approved,2026-01-01 11:00:21.829,Ana Silva
9,LOTE-SK-M-SOP-013-8350,LOTE-M-SOP-013-8350,WO-9834,2026-01-01,Shift 1,SS-002,SK-FR-013-MDPE-500-2C,FR-013-MDPE-500,2,"COR-002, COR-005",Centering,Critical,0.65,ISO 2859-1,II,Every 30 min,Registration Gauge,7562,L,200,2,3,0,Approved,2026-01-01 11:06:54.650,Ana Silva



fact_ink_disposition_lot_cq_raw -- first 30 rows


,PrintLot,BottleLot,WorkOrder,ProductionDate,Shift,MachineId,PrintToolId,BottleId,ColorCount,ColorsUsed,LotSize,CodeLetter,SampleSize,CriticalDefects,MajorDefects,MinorDefects,TotalSampleDefects,AttributesDecision,FinalLotDecision,LotDecisionDateTime,Remarks,Inspector
0,LOTE-SK-M-SOP-011-3240,LOTE-M-SOP-011-3240,WO-9323,2026-01-01,Shift 3,SS-001,SK-FR-011-PET-400-2C,FR-011-PET-400,2,"COR-006, COR-008",19128,M,315,0,0,0,0,Approved,Approved,2026-01-02 00:09:57.057,—,Elena Santos
1,LOTE-SK-M-SOP-013-8350,LOTE-M-SOP-013-8350,WO-9834,2026-01-01,Shift 1,SS-002,SK-FR-013-MDPE-500-2C,FR-013-MDPE-500,2,"COR-002, COR-005",7562,L,200,0,0,0,0,Approved,Approved,2026-01-01 16:22:39.174,—,Ana Silva
2,LOTE-SK-M-SOP-009-8360,LOTE-M-SOP-009-8360,WO-9835,2026-01-01,Shift 2,SS-002,SK-FR-009-PET-350-1C,FR-009-PET-350,1,COR-008,16871,M,315,0,0,0,0,Approved,Approved,2026-01-02 12:27:20.726,—,Diogo Ferreira
3,LOTE-SK-M-SOP-004-3250,LOTE-M-SOP-004-3250,WO-9324,2026-01-02,SHIFT 3,SS-001,SK-FR-004-rPET-330-1C,FR-004-rPET-330,1,COR-008,18754,M,315,0,0,0,0,Approved,Approved,2026-01-02 21:38:13.807,—,Beatriz Costa
4,LOTE-SK-M-SOP-004-3260,LOTE-M-SOP-004-3260,WO-9325,2026-01-02,Shift 2,SS-001,SK-FR-004-rPET-330-1C,FR-004-rPET-330,1,COR-008,8428,L,200,0,0,0,0,Approved,Approved,2026-01-03 06:30:26.901,—,Elena Santos
5,LOTE-SK-M-SOP-009-8370,LOTE-M-SOP-009-8370,WO-9836,2026-01-02,Shift 1,SS-002,SK-FR-009-PET-350-1C,FR-009-PET-350,1,COR-008,18243,M,315,0,0,0,0,Approved,Approved,2026-01-03 09:47:44.851,—,Elena Santos
6,LOTE-SK-M-SOP-004-3270,LOTE-M-SOP-004-3270,WO-9326,2026-01-03,Shift 3,SS-001,SK-FR-004-rPET-330-1C,FR-004-rPET-330,1,COR-008,6660,L,200,0,0,0,0,Approved,Approved,2026-01-03 13:29:01.577,—,Carlos Mendes
7,LOTE-SK-M-SOP-014-3280,LOTE-M-SOP-014-3280,WO-9327,2026-01-03,Shift 1,SS-001,SK-FR-014-PETG-500-3C,FR-014-PETG-500,3,"COR-006, COR-008",7556,L,200,0,0,0,0,Approved,Approved,2026-01-03 22:55:52.367,—,Carlos Mendes
8,LOTE-SK-M-SOP-007-8380,LOTE-M-SOP-007-8380,WO-9837,2026-01-03,Shift 1,SS-002,SK-FR-007-HDPE-PCR-350-2C,FR-007-HDPE-PCR-350,2,"COR-002, COR-003",10917,M,315,0,0,0,0,Approved,Approved,2026-01-03 22:26:23.830,—,Beatriz Costa
9,LOTE-SK-M-SOP-014-3290,LOTE-M-SOP-014-3290,WO-9328,2026-01-05,Shift 1,SS-001,SK-FR-014-PETG-500-3C,FR-014-PETG-500,3,"COR-002, COR-008",6738,L,200,0,0,0,0,Approved,Approved,2026-01-05 14:51:06.096,—,Carlos Mendes


In [9]:
for name, df in cq_tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"{name}: {len(df):,} rows, {nulls.sum():,} missing values across {len(nulls)} columns")


fact_cap_inspection_variable_cq_raw: 83,947 rows, 8,270 missing values across 5 columns
fact_cap_attribute_inspection_cq_raw: 9,924 rows, 0 missing values across 0 columns
fact_cap_disposition_lot_cq_raw: 1,654 rows, 0 missing values across 0 columns


fact_bottle_inspection_variables_cq_raw: 124,580 rows, 4,380 missing values across 2 columns
fact_bottle_attribute_inspection_cq_raw: 17,520 rows, 0 missing values across 0 columns
fact_bottle_disposition_lot_cq_raw: 2,190 rows, 0 missing values across 0 columns
fact_ink_attribute_inspection_cq_raw: 4,456 rows, 0 missing values across 0 columns
fact_ink_disposition_lot_cq_raw: 557 rows, 0 missing values across 0 columns


## 3. Dimension / master-data tables

Governed reference data (machine capacities, cap/bottle specifications,
control plans). These come from engineering, not the shop floor, so we
expect -- and will verify -- that they arrive much cleaner than the fact
tables.


In [10]:
dim_files = {
    'dim_masterbatch': f'{DIM}/dim_masterbatch.csv',
    'dim_machine_setup': f'{DIM}/dim_machine_setup.csv',
    'dim_cap': f'{DIM}/dim_cap.csv',
    'dim_cap_control_plan_cq': f'{DIM}/dim_cap_control_plan_cq.csv',
    'dim_bottle_control_plan_cq': f'{DIM}/dim_bottle_control_plan_cq.csv',
    'dim_ink_control_plan_cq': f'{DIM}/dim_ink_control_plan_cq.csv',
}
dim_tables = {name: pd.read_csv(path, encoding='utf-8-sig') for name, path in dim_files.items()}
for name, df in dim_tables.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns, "
          f"{int(df.isna().sum().sum())} missing values")
    display(df.head(10))


dim_masterbatch: 132 rows x 9 columns, 0 missing values

,ProductId,ProductType,MoldId,BaseMaterial,ColorId,ColorName,PantoneCodeApprox,MasterbatchType,StandardDosagePctMass
0,FR-001-HDPE-300,Bottle,M-SOP-001,HDPE,COR-007,Cardinal Red,Pantone 200 C,Red Masterbatch,0.020
1,FR-002-HDPE-PCR-300,Bottle,M-SOP-002,HDPE-PCR,COR-003,Cobalt Blue,Pantone 2145 C,Blue Masterbatch,0.020
2,FR-001-HDPE-PCR-300,Bottle,M-SOP-001,HDPE-PCR,COR-007,Cardinal Red,Pantone 200 C,Red Masterbatch,0.020
3,FR-001-rPET-300,Bottle,M-SOP-001,rPET,COR-008,Natural (Uncolored),—,No Masterbatch,0.000
4,FR-003-rPET-300,Bottle,M-SOP-003,rPET,COR-008,Natural (Uncolored),—,No Masterbatch,0.000
5,FR-004-rPET-330,Bottle,M-SOP-004,rPET,COR-008,Natural (Uncolored),—,No Masterbatch,0.000
6,FR-008-HDPE-350,Bottle,M-SOP-008,HDPE,COR-006,Graphite Gray,Pantone Cool Gray 11,Gray Masterbatch,0.020
7,FR-007-HDPE-350,Bottle,M-SOP-007,HDPE,COR-005,Translucent Amber,Pantone 1375 C,Amber Masterbatch,0.015
8,FR-009-HDPE-350,Bottle,M-SOP-009,HDPE,COR-007,Cardinal Red,Pantone 200 C,Red Masterbatch,0.020
9,FR-008-HDPE-PCR-350,Bottle,M-SOP-008,HDPE-PCR,COR-003,Cobalt Blue,Pantone 2145 C,Blue Masterbatch,0.020


dim_machine_setup: 38 rows x 8 columns, 0 missing values


,MoldId,MachineId,Product,Cavities,RatedCapacityPerDay,RatedCapacityPerHour,CyclesPerHour,IdealCycleTimeSec
0,M-INJ-001,IM-001,Tampa rosca comum 24/410,24,~28.800 pieces/day,~1200 pieces/h,50,72
1,M-INJ-002,IM-002,Tampa rosca comum 24/410,16,~19.200 pieces/day,~800 pieces/h,50,72
2,M-INJ-003,IM-003,Tampa rosca comum 24/415,32,~38.400 pieces/day,~1600 pieces/h,50,72
3,M-INJ-004,IM-004,Tampa rosca comum 28/410,24,~28.800 pieces/day,~1200 pieces/h,50,72
4,M-INJ-005,IM-005,Tampa flip top 24/410,16,~19.200 pieces/day,~800 pieces/h,50,72
5,M-INJ-009,IM-006,Tampa flip top 28/415,12,~14.400 pieces/day,~600 pieces/h,50,72
6,M-INJ-006,IM-006,Tampa flip top 24/415,8,~9.600 pieces/day,~400 pieces/h,50,72
7,M-INJ-007,IM-004,Tampa flip top 28/410,12,~14.400 pieces/day,~600 pieces/h,50,72
8,M-INJ-008,IM-005,Tampa flip top 28/410,16,~19.200 pieces/day,~800 pieces/h,50,72
9,M-SOP-001,ISBM-001,"Frasco Round, 300 ml",8,~14.400 bottles/day,~600 bottles/h,75,48


dim_cap: 31 rows x 15 columns, 0 missing values


...

,Process,Operation,Characteristic,Class,InspectionType,Specification,Method,Equipment,Standard,ISOLevel,AQL,Frequency,LotSize,ISOCode,SampleSize,AcceptanceNumber,RejectionNumber,Owner,ReactionPlan
0,Injection Molding,Production,Weight,Major,Variable,Per technical data sheet,Weighing,Scale 0.001 g,ISO 3951,—,—,Every 30 min,Production run,—,10 pieces,—,—,Quality,Adjust Process
1,Injection Molding,Production,Height,Major,Variable,Per drawing,Caliper,Caliper,ISO 3951,—,—,Every 1 h,Production run,—,10 pieces,—,—,Quality,Adjust Mold
2,Injection Molding,Production,Diameter,Major,Variable,Per drawing,Caliper,Caliper,ISO 3951,—,—,Every 1 h,Production run,—,10 pieces,—,—,Quality,Adjust Mold
3,Injection Molding,Production,Torque,Critical,Variable,Per specification,Torque Test,Torque Wrench,ISO 3951,—,—,Per lot,Lot,—,5 pieces,—,—,Laboratory,Adjust Mold
4,Injection Molding,Production,Thread,Critical,Attribute,Per drawing,Go/No-Go,Gauge,ISO 2859-1,II,0.1,Per lot,Per lot,Auto,Auto,0,1,Quality,Block Lot
5,Injection Molding,Production,Sealing,Critical,Attribute,No leakage,Leak Test,Leak Tester,ISO 2859-1,II,0.1,Per lot,Per lot,Auto,Auto,0,1,Laboratory,Block Lot
6,Injection Molding,Production,Flash,Major,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Machine
7,Injection Molding,Production,Short Shot,Major,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Pressure
8,Injection Molding,Production,Stain,Minor,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,1.5,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Process
9,Injection Molding,Production,Color,Minor,Attribute,Standard,Visual,D65 Light Booth,ISO 2859-1,II,1.5,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Quality,Change Masterbatch Lot


dim_bottle_control_plan_cq: 13 rows x 19 columns, 0 missing values


,Process,Operation,Characteristic,Class,InspectionType,Specification,Method,Equipment,Standard,ISOLevel,AQL,Frequency,LotSize,ISOCode,SampleSize,AcceptanceNumber,RejectionNumber,Owner,ReactionPlan
0,Blow Molding,Production,Weight,Major,Variable,Per technical data sheet,Weighing,Scale 0.01 g,ISO 3951,—,—,Every 30 min,Production run (30min),—,5 pieces,—,—,Quality,Adjust Parameters
1,Blow Molding,Production,Height,Major,Variable,Per drawing,Caliper,Caliper,ISO 3951,—,—,Every 1 h,Production run (1h),—,5 pieces,—,—,Quality,Adjust Mold
2,Blow Molding,Production,Neck Diameter,Critical,Variable,Per drawing,Go/No-Go,Gauge,ISO 3951,—,—,Every 1 h,Production run (1h),—,5 pieces,—,—,Quality,Block Lot
3,Blow Molding,Production,Thickness,Major,Variable,Per technical data sheet,Ultrasound,Thickness Gauge,ISO 3951,—,—,Every 2 h,Production run (2h),—,5 pieces,—,—,Quality,Adjust Process
4,Blow Molding,Production,Overflow Volume,Major,Variable,Per technical data sheet,Fill Test,Graduated Cylinder,ISO 3951,—,—,Per lot,Lot,—,3 pieces,—,—,Laboratory,Block Lot
5,Blow Molding,Production,Leakage,Critical,Attribute,No leakage,Leak Test,Leak Tester,ISO 2859-1,II,0.1,Per lot,Per lot,Auto,Auto,0,1,Laboratory,Block Lot
6,Blow Molding,Production,Flash,Major,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Mold
7,Blow Molding,Production,Bubbles,Major,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Segregate
8,Blow Molding,Production,Black Specks,Minor,Attribute,≤1 per piece,Visual,D65 Light Booth,ISO 2859-1,II,1.5,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Cleaning
9,Blow Molding,Production,Stain,Minor,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,1.5,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Process


dim_ink_control_plan_cq: 8 rows x 19 columns, 0 missing values


,Process,Operation,Characteristic,Class,InspectionType,Specification,Method,Equipment,Standard,ISOLevel,AQL,Frequency,LotSize,ISOCode,SampleSize,AcceptanceNumber,RejectionNumber,Owner,ReactionPlan
0,Screen Printing,Production,Registration,Critical,Attribute,Position per gauge,Visual,Registration Gauge,ISO 2859-1,II,0.65,Setup + 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Screen
1,Screen Printing,Production,Centering,Critical,Attribute,Centered ±1 mm,Visual,Registration Gauge,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Registration
2,Screen Printing,Production,Coverage,Major,Attribute,100% printed area,Visual,D65 Light Booth,ISO 2859-1,II,0.65,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Ink/Pressure
3,Screen Printing,Production,Color,Minor,Attribute,ΔE ≤ 1.5 vs standard,Colorimeter,Colorimeter,ISO 2859-1,II,1.50,Every 30 min,Per lot,Auto,Auto,ver tabela,ver tabela,Quality,Adjust Ink Formula
4,Screen Printing,Production,Smudge,Minor,Attribute,Absent,Visual,D65 Light Booth,ISO 2859-1,II,1.50,Continuous,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Adjust Speed
5,Screen Printing,Production,Pinholes,Minor,Attribute,Absent,Visual,Lupa 10x,ISO 2859-1,II,1.50,Continuous,Per lot,Auto,Auto,ver tabela,ver tabela,Operator,Clean Screen
6,Screen Printing,Production,Adhesion,Critical,Attribute,Class ≥ 4B (ASTM D3359),Cross Hatch,ASTM D3359 Kit,ISO 2859-1,II,0.10,Per lot,3 samples,—,3,0,1,Laboratory,Block Lot
7,Screen Printing,Production,Cure,Critical,Attribute,No removal after 50 rubs,MEK Rub Test,MEK Rub Test,ISO 2859-1,II,0.10,Per lot,3 samples,—,3,0,1,Laboratory,Reprocess (UV Re-cure)


## 4. Problems found -- summary

*"The data arrives like this -- raw. Look at the problems:"*

Running the three commands above on every table surfaces the same handful
of recurring, realistic data-entry problems:

- **Missing values** -- e.g. a handful of blank `OperatorId` / `MaintenanceTechnician`
  entries where a shift report wasn't fully filled in.
- **Inconsistent categories** -- the same process typed multiple ways in
  the same column: `Injection Molding`, `injection molding`,
  `INJECTION MOLDING`, `  Injection Molding  ` (stray whitespace). A naive
  `GROUP BY Process` would currently split what is really *one* process
  into several phantom categories.
- **Negative quantities** -- a small number of `RejectedQty` (rejected
  pieces) and `EndWeightKg` (remaining material weight) values are
  negative. Physically impossible: nobody rejects "-12" caps. This is a
  sign-entry error, most likely a scrap sheet transcribed with a stray
  minus sign.
- **Duplicated rows** -- a small fraction of fully identical rows exist in
  every fact table, consistent with a double MES submission or a re-run
  export job that appended instead of overwriting.
- **Placeholder nulls disguised as text** -- some text fields contain a
  single space, a dash (`-`, `--`, `---`), or a slash (`/`, `//`) where an
  operator meant "nothing to report" instead of leaving the cell empty.
  These are *not* legitimate categories and must not be treated as such
  (e.g. they must not show up as a phantom `"-"` maintenance technician in
  a report).

None of this is fixed here -- see `02_data_cleaning_production.ipynb` for
the cleaning strategy and the reasoning behind each choice.
